# How big is MoM-BH\*-1?

**Goal:** estimate the size of MoM-BH\*-1's glowing surface (its "photosphere"), which is one of the open questions about this object.

**The idea:** it's the same physics you'd use for a star. A glowing sphere at temperature T looks brighter the bigger it is. So if we know:
1. how bright it looks at one wavelength (from your blackbody fit),
2. its temperature (about 5,000–5,500 K from your two measurements),
3. its distance (from its redshift),

then there is only one radius that makes it look exactly as bright as it does. The Stefan–Boltzmann law, L = 4πR²σT⁴, then gives its total luminosity, which you can check against the paper.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**.

*Data: Naidu et al., Zenodo record 15059215 (CC BY 4.0).*

## Step 1 — Download the spectrum

In [ ]:
!wget -q -O spec_150135.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_150135.spec.fits?download=1"

import os
size_kb = os.path.getsize("spec_150135.fits") / 1024 if os.path.exists("spec_150135.fits") else 0
print(f"spec_150135.fits: {size_kb:.0f} KB ->", "OK" if size_kb > 300 else "download failed, run this cell again in a few minutes")

## Step 2 — Redo the blackbody fit

This is the same fit as in the temperature notebook, so you should get the same answer: about 4,970 K, with a brightness of about 298 nJy at 5,000 Å. The brightness at 5,000 Å is the number we need for the size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

Z = 7.7569
FIT_START = 4150
LINE_MASKS = [(4290, 4400, "Hγ"), (4770, 5040, "Hβ + [O III]"), (5830, 5930, "He I")]
HC_OVER_K = 1.4388e8   # Å·K


def load_spectrum(name):
    '''Return wavelength (µm), flux and error (µJy).'''
    with fits.open(name) as f:
        ext = "SPEC1D" if "SPEC1D" in f else 1
        tab = Table(f[ext].data)
    cols = {c.lower(): c for c in tab.colnames}
    err_col = cols.get("err", cols.get("full_err"))
    wave = np.array(tab[cols["wave"]], dtype=float)
    flux = np.array(tab[cols["flux"]], dtype=float)
    err = np.array(tab[err_col], dtype=float)
    if np.nanmedian(wave) > 100:
        wave = wave / 1e4
    good = np.isfinite(wave) & np.isfinite(flux) & np.isfinite(err) & (err > 0)
    return wave[good], flux[good], err[good]


def blackbody_shape(lam, T, lam_ref=5000.0):
    '''Blackbody B_nu shape at wavelength lam (Å), scaled to equal 1 at lam_ref.'''
    x = HC_OVER_K / (lam * T)
    x_ref = HC_OVER_K / (lam_ref * T)
    return (lam_ref / lam) ** 3 * np.expm1(x_ref) / np.expm1(x)


wave, flux, err = load_spectrum("spec_150135.fits")
rest = wave * 1e4 / (1 + Z)
flux_nJy, err_nJy = flux * 1000, err * 1000

use = rest >= FIT_START
for lo, hi, name in LINE_MASKS:
    use &= ~((rest >= lo) & (rest <= hi))
lam, f, e = rest[use], flux_nJy[use], err_nJy[use]
w = 1 / e**2

temps = np.arange(2500, 15001, 10)
chi2 = np.empty(temps.size)
heights = np.empty(temps.size)
for i, T in enumerate(temps):
    m = blackbody_shape(lam, T)
    heights[i] = np.sum(w * f * m) / np.sum(w * m * m)
    chi2[i] = np.sum(w * (f - heights[i] * m) ** 2)

best = np.argmin(chi2)
T_best, A_best = int(temps[best]), heights[best]
print(f"Best-fit temperature: {T_best} K")
print(f"Brightness of the fitted blackbody at 5000 Å (rest frame): {A_best:.0f} nJy")

## Step 3 — Turn brightness and temperature into a size

**The physics, in plain words:**
- Each square centimetre of a blackbody surface at temperature T gives off a fixed amount of light, set by the Planck formula, B_ν(T).
- A sphere of radius R therefore gives off 4πR² × πB_ν(T) at each frequency.
- We receive that light spread over a huge sphere around the object. At cosmological distances we use the **luminosity distance**, D_L, and include one extra factor of (1 + z) for how redshift stretches the frequencies.

Putting it together:

**observed brightness = (1 + z) × π B_ν(T) × R² / D_L²**, so **R = D_L × √( observed brightness / ((1 + z) π B_ν(T)) )**

Then the **Stefan–Boltzmann law** gives the total luminosity: **L = 4πR²σT⁴**.

We do this for three temperatures:
- **your spectrum fit** (about 5,000 K);
- **your MIRI/F444W colour** (5,500 K);
- **Curtis et al.'s atmosphere model** (6,500 K), to see how their temperature would change the size.

The **light-crossing time** (R ÷ speed of light) is how long light takes to cross the glowing surface. It sets the fastest the whole object can change brightness together.

In [ ]:
from astropy.cosmology import Planck18
from astropy import units as u, constants as const
from astropy.modeling.models import BlackBody

D_L = Planck18.luminosity_distance(Z).to(u.cm)     # luminosity distance
brightness = A_best * u.nJy                         # observed brightness at rest-frame 5000 Å


def size_and_luminosity(T):
    '''Radius of the glowing surface and total luminosity for temperature T (kelvin).'''
    B = BlackBody(temperature=T * u.K)(5000 * u.AA)                  # Planck formula at 5000 Å
    R = (D_L * np.sqrt(brightness / ((1 + Z) * np.pi * u.sr * B))).to(u.cm)
    L = (4 * np.pi * R**2 * const.sigma_sb * (T * u.K) ** 4).to(u.erg / u.s)
    return R, L


print(f"Luminosity distance: {D_L.to(u.Gpc):.1f}\n")
print(f"{'Temperature':>12} {'Radius (AU)':>12} {'Radius (Suns)':>14} {'Luminosity (erg/s)':>19} {'light-crossing':>15}")
results = {}
for T, label in [(T_best, "your spectrum fit"), (5500, "your MIRI/F444W colour"), (6500, "Curtis et al. temperature")]:
    R, L = size_and_luminosity(T)
    results[T] = (R, L)
    print(f"{T:>10} K {R.to(u.AU).value:>12,.0f} {(R / const.R_sun).decompose().value:>14,.0f} "
          f"{L.value:>19.2e} {(R / const.c).to(u.day).value:>11.1f} days   ({label})")

## Step 4 — Compare with published numbers

- **Luminosity:** the paper integrated its model spectrum and got about 10^44.5 ≈ 3 × 10^44 erg/s. If your number is close, your simple blackbody picture captures the object's total energy output.
- **The paper's "10–100 AU":** this is **not** the size of the glowing surface. It is the **thickness** of the dense gas in their model: column density ÷ density (10^25.8 ÷ 10^11 cm). The cell below works it out.
- **RX1**, a lensed little red dot whose brightness changes revealed a glowing surface of about **2,000 AU** (Zhang et al.), measured with a completely independent method on a different object.

In [ ]:
R_mine, L_mine = results[T_best]
layer = (10**25.8 / 10**11 * u.cm).to(u.AU)          # column density / density = thickness

print(f"Your luminosity:        {L_mine.value:.2e} erg/s  (log = {np.log10(L_mine.value):.2f})")
print(f"Paper's luminosity:     about 3.2e+44 erg/s  (log = 44.5)")
print(f"\nYour radius:            {R_mine.to(u.AU).value:,.0f} AU (at {T_best} K), "
      f"{results[5500][0].to(u.AU).value:,.0f} AU (at 5500 K)")
print(f"Dense gas layer in the paper's model: {layer.value:.0f} AU thick")
print(f"RX1 (lensed little red dot):          about 2,000 AU")
print(f"\nLight-crossing time of your surface: {(R_mine / const.c).to(u.day).value:.0f} days "
      f"(the paper's possible brightening happened over about 56 days, rest frame)")

## Step 5 — See the sizes side by side

The scale is **logarithmic**: each step to the right is 10 times bigger. The red bar is your result; its black line spans your two temperatures (5,000–5,500 K). The plot is saved as **MoM-BH1_size.png**.

In [ ]:
sun = (const.R_sun).to(u.AU).value
objects = [
    ("The Sun", sun, "lightgray"),
    ("Largest known stars (~1,500 Suns)", 1500 * sun, "lightgray"),
    ("Neptune's orbit", 30.1, "lightgray"),
    ("Dense gas layer in the paper's model", layer.value, "tab:blue"),
    ("MoM-BH*-1 glowing surface (your result)", R_mine.to(u.AU).value, "tab:red"),
    ("RX1, a lensed little red dot", 2000, "tab:purple"),
]

fig, ax = plt.subplots(figsize=(10, 4.5))
names = [o[0] for o in objects]
sizes = [o[1] for o in objects]
ax.barh(names, sizes, color=[o[2] for o in objects])
lo_hi = sorted([results[5500][0].to(u.AU).value, R_mine.to(u.AU).value])
ax.plot(lo_hi, [4, 4], color="black", lw=2)        # range across your two temperatures (row 4)
for i, s in enumerate(sizes):
    label = f"{s:,.0f} AU" if s >= 10 else f"{s:.3g} AU"
    ax.text(s * 1.3, i, label, va="center", fontsize=9)
ax.set_xscale("log")
ax.set_xlim(1e-3, 3e4)
ax.set_xlabel("Radius (AU, logarithmic scale)")
ax.set_title("How big is MoM-BH*-1?")
ax.invert_yaxis()
plt.tight_layout()
fig.savefig("MoM-BH1_size.png", dpi=150)
plt.show()

## What your result means

- **If your luminosity matches the paper's (about 3 × 10^44 erg/s),** a simple blackbody surface accounts for the object's total energy output. Two independent routes then agree: your blackbody fit, and the paper's gas model.
- **Your radius is the size of the glowing surface; the paper's 10–100 AU is the thickness of the dense gas.** They measure different things, so a big difference doesn't mean either is wrong. For example, the dense gas could be a thin layer in a much larger envelope. How the two pictures fit together is a genuine open question.
- **Compare with RX1.** A similar size from a completely different method and object suggests this may be a common feature of black hole stars.

**Caveats** (mention these if you write it up):
- This assumes a round, blackbody-like surface. You found the spectrum is close to, but not exactly, a blackbody.
- **Temperature is the biggest uncertainty.** A hotter surface needs to be smaller to give the same brightness, as the 6,500 K row shows.
- The object seems to have brightened about 30% between 2023 and 2024. Radius goes as the square root of brightness, so that's about a 15% effect.
- Prism spectra can miss a little light at the slit edges, so the true brightness and size could be slightly larger.